# Cours 4 : Web Scraping Avancé - AlloCiné - CORRIGÉ

## Objectifs
- Scraper plusieurs pages d'un site web
- Extraire et sauvegarder des liens dans un CSV
- Réimporter un CSV et transformer une colonne en liste
- Extraire des informations techniques en gérant des structures HTML variables

In [2]:
# Imports nécessaires
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random
import time

---
# PARTIE 1 : Récupérer les liens des films sur plusieurs pages

**Objectif** : Scraper les 3 premières pages des films à l'affiche sur AlloCiné et sauvegarder tous les liens.

**URL cible** : `https://www.allocine.fr/film/aucinema/?page=1`

---
## Étape 1.1 : Comprendre la pagination

**Ce qu'il faut faire :**
1. Observer l'URL : `https://www.allocine.fr/film/aucinema/?page=1`
2. Remarquer le paramètre `?page=1` qui change selon la page
3. Page 1 → `?page=1`, Page 2 → `?page=2`, etc.

**Question** : Comment construire dynamiquement l'URL pour n'importe quelle page ?

In [3]:
# Étape 1.1 : Construire l'URL de base et tester avec page=1

# URL de base (sans le numéro de page)
url_base = 'https://www.allocine.fr/film/aucinema/?page='

# Construire l'URL pour la page 1
numero_page = 1
url = url_base + str(numero_page)

# Alternative avec f-string (plus élégant)
url = f'https://www.allocine.fr/film/aucinema/?page={numero_page}'

print(f'URL construite : {url}')

URL construite : https://www.allocine.fr/film/aucinema/?page=1


---
## Étape 1.2 : Se connecter à la page 1 et parser le HTML

**Ce qu'il faut faire :**
1. Définir les headers avec un User-Agent
2. Utiliser `requests.get()` pour récupérer la page
3. Vérifier avec `response.ok`
4. Parser avec BeautifulSoup

In [4]:
# Étape 1.2 : Connexion et parsing

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)

if response.ok:
    print('Connexion réussie !')
    soup = BeautifulSoup(response.text, 'lxml')
else:
    print(f'Erreur : Code {response.status_code}')

Connexion réussie !


---
## Étape 1.3 : Trouver la section contenant les films

**Ce qu'il faut faire :**
1. Utiliser l'inspecteur du navigateur pour trouver la structure
2. La section principale a la classe `"section section-wrap gd-2-cols gd-gap-30"`
3. À l'intérieur, chercher les `<ul>` puis les `<li class="mdl">`

In [5]:
# Étape 1.3 : Trouver la section principale

section_principale = soup.find('section', class_='section section-wrap gd-2-cols gd-gap-30')

if section_principale:
    print('Section principale trouvée !')
    print(section_principale.prettify())
else:
    print('Section non trouvée...')

Section principale trouvée !
<section class="section section-wrap gd-2-cols gd-gap-30">
 <div class="gd-col-left">
  <div class="js-nativebanner ad-nativebanner ad-item" id="dfp-nativebanner">
  </div>
  <div class="dropdown-custom-holder dropdown-custom-alter">
   <span class="what">
    Trier par
   </span>
   <div class="dropdown-custom js-dropdown">
    <span class="title">
     Nombre de cinémas
    </span>
    <i class="btn">
    </i>
    <ul class="list">
     <li class="item">
      <span class="ACrL2ZACrpbG0vYXVjaW5lbWEv item-content">
       Nombre de cinémas
      </span>
     </li>
     <li class="item">
      <span class="ACrL2ZACrpbG0vYXVjaW5lbWEvZGF0ZS1zb3J0aWUv item-content">
       Date de sortie
      </span>
     </li>
    </ul>
   </div>
  </div>
  <ul>
   <li class="mdl">
    <style>
     :root {
                --customAnchorPlusTxtColor: #ffffff;
        }
    
    /* Mobile */
    .anchor-plus {
        background-image: url(https://fr.web.img6.acsta.net/img/d0/

In [6]:
subsection = section_principale.find('div', class_='gd-col-left')
print(subsection)

<div class="gd-col-left">
<div class="js-nativebanner ad-nativebanner ad-item" id="dfp-nativebanner"></div>
<div class="dropdown-custom-holder dropdown-custom-alter">
<span class="what">Trier par</span>
<div class="dropdown-custom js-dropdown">
<span class="title">Nombre de cinémas</span>
<i class="btn"></i>
<ul class="list">
<li class="item">
<span class="ACrL2ZACrpbG0vYXVjaW5lbWEv item-content">
            Nombre de cinémas
        </span>
</li>
<li class="item">
<span class="ACrL2ZACrpbG0vYXVjaW5lbWEvZGF0ZS1zb3J0aWUv item-content">
            Date de sortie
        </span>
</li>
</ul>
</div>
</div>
<ul>
<li class="mdl">
<style>
            :root {
                --customAnchorPlusTxtColor: #ffffff;
        }
    
    /* Mobile */
    .anchor-plus {
        background-image: url(https://fr.web.img6.acsta.net/img/d0/0c/d00c6a2ed8701a68ba98e7a05f91f50a.jpg);
    }

    /* Desktop/Tablet */
    .anchor-plus::before {
        background-image: url(https://fr.web.img3.acsta.net/img/55/

---
## Étape 1.4 : Extraire les films (éléments `<li>`)

**Ce qu'il faut faire :**
1. Trouver toutes les listes `<ul>` dans la section
2. La liste des films est généralement la 2ème `<ul>` (index 1)
3. Extraire tous les `<li class="mdl">` de cette liste

In [7]:
# Étape 1.4 : Extraire les éléments li des films

# Trouver toutes les listes ul dans la section
listes_ul = subsection.find_all('ul')
print(f'Nombre de listes <ul> trouvées : {len(listes_ul)}')

# La liste des films est la 2ème (index 1)
liste_films = listes_ul[1]

# Extraire tous les li avec la classe "mdl"
films_li = liste_films.find_all('li', class_='mdl')
print(f'Nombre de films trouvés : {len(films_li)}')

Nombre de listes <ul> trouvées : 2
Nombre de films trouvés : 16


---
## Étape 1.5 : Extraire le titre et le lien d'un film

**Ce qu'il faut faire :**
1. Pour un film (un `<li>`), trouver le `<div class="meta">`
2. À l'intérieur, trouver le `<a class="meta-title-link">`
3. Récupérer le texte (titre) avec `.get_text(strip=True)`
4. Récupérer le lien avec `['href']` et construire l'URL complète

In [6]:
# Étape 1.5 : Extraire titre et lien pour UN film (test)

# Prendre le premier film pour tester
film_test = films_li[0]

# Trouver la div meta
div_meta = film_test.find('div', class_='meta')

# Trouver le lien avec le titre
lien_titre = div_meta.find('a', class_='meta-title-link')

# Extraire le titre
titre = lien_titre.get_text(strip=True)

# Extraire et construire l'URL complète
url_relative = lien_titre['href']
url_complete = 'https://www.allocine.fr' + url_relative

print(f'Titre : {titre}')
print(f'Lien : {url_complete}')

Titre : L’Affaire Bojarski
Lien : https://www.allocine.fr/film/fichefilm_gen_cfilm=1000014092.html


---
## Étape 1.6 : Boucler sur tous les films d'une page

**Ce qu'il faut faire :**
1. Créer deux listes vides : `liste_titres` et `liste_liens`
2. Boucler sur tous les `<li>` de la page
3. Pour chaque film, extraire titre et lien
4. Ajouter aux listes

In [7]:
# Étape 1.6 : Boucler sur tous les films d'une page

liste_titres = []
liste_liens = []

for film in films_li:
    # Trouver la div meta
    div_meta = film.find('div', class_='meta')
    
    if div_meta:
        # Trouver le lien
        lien_titre = div_meta.find('a', class_='meta-title-link')
        
        if lien_titre:
            titre = lien_titre.get_text(strip=True)
            lien = 'https://www.allocine.fr' + lien_titre['href']
            
            liste_titres.append(titre)
            liste_liens.append(lien)

print(f'Films extraits : {len(liste_titres)}')
print(f'\nPremiers titres :')
for i in range(min(5, len(liste_titres))):
    print(f'  - {liste_titres[i]}')

Films extraits : 15

Premiers titres :
  - L’Affaire Bojarski
  - La Femme de ménage
  - Le Mage du Kremlin
  - Zootopie 2
  - Gourou


---
## Étape 1.7 : Boucler sur 3 pages

**Ce qu'il faut faire :**
1. Créer deux listes globales `tous_titres` et `tous_liens`
2. Boucler sur les pages 1, 2, 3 avec `for page in range(1, 4)`
3. Pour chaque page :
   - Construire l'URL
   - Se connecter et parser
   - Trouver la section et les films
   - Extraire titres et liens
   - Ajouter aux listes globales avec `.extend()`
4. **Important** : Ajouter `time.sleep(1)` entre chaque page

In [8]:
# Étape 1.7 : Boucler sur 3 pages (tout le code dans une seule boucle)

# Listes globales pour stocker tous les résultats
tous_titres = []
tous_liens = []

# Headers (définis une seule fois)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# Boucler sur les pages 1, 2, 3
for page in range(1, 4):
    print(f'Scraping page {page}...')
    
    # 1. Construire l'URL
    url = f'https://www.allocine.fr/film/aucinema/?page={page}'
    
    # 2. Se connecter
    response = requests.get(url, headers=headers)
    
    if not response.ok:
        print(f'  Erreur page {page}')
        continue
    
    # 3. Parser le HTML
    soup = BeautifulSoup(response.text, 'lxml')
    
    # 4. Trouver la section
    section = soup.find('section', class_='section section-wrap gd-2-cols gd-gap-30')
    
    if not section:
        print(f'  Section non trouvée')
        continue
    
    # 5. Trouver la liste des films
    listes_ul = section.find_all('ul')
    
    if len(listes_ul) < 2:
        print(f'  Liste de films non trouvée')
        continue
    
    films_li = listes_ul[1].find_all('li', class_='mdl')
    
    # 6. Extraire titres et liens de cette page
    titres_page = []
    liens_page = []
    
    for film in films_li:
        div_meta = film.find('div', class_='meta')
        if div_meta:
            lien_titre = div_meta.find('a', class_='meta-title-link')
            if lien_titre:
                titres_page.append(lien_titre.get_text(strip=True))
                liens_page.append('https://www.allocine.fr' + lien_titre['href'])
    
    # 7. Ajouter aux listes globales
    tous_titres.extend(titres_page)
    tous_liens.extend(liens_page)
    
    print(f'  → {len(titres_page)} films récupérés')
    
    # 8. Pause entre les pages
    time.sleep(1)

print(f'\nTotal : {len(tous_titres)} films récupérés sur 3 pages')

Scraping page 1...
  → 15 films récupérés
Scraping page 2...
  → 15 films récupérés
Scraping page 3...
  → 15 films récupérés

Total : 45 films récupérés sur 3 pages


---
## Étape 1.8 : Créer le DataFrame et exporter en CSV

**Ce qu'il faut faire :**
1. Créer un DataFrame avec les colonnes `'titre'` et `'lien'`
2. Afficher les premières lignes pour vérifier
3. Exporter en CSV avec `to_csv('films.csv', index=False)`

In [9]:
# Étape 1.8 : Créer DataFrame et exporter

df_films = pd.DataFrame({
    'titre': tous_titres,
    'lien': tous_liens
})

print('Aperçu du DataFrame :')
print(df_films.head(10))

# Exporter en CSV
df_films.to_csv('films.csv', index=False)
print(f'\nFichier "films.csv" créé avec {len(df_films)} films !')

Aperçu du DataFrame :
                           titre  \
0             L’Affaire Bojarski   
1             La Femme de ménage   
2             Le Mage du Kremlin   
3                     Zootopie 2   
4                         Gourou   
5  Avatar : de Feu et de Cendres   
6            Le Chant des forêts   
7                Furcy, né libre   
8                       Ma frère   
9                Les Légendaires   

                                                lien  
0  https://www.allocine.fr/film/fichefilm_gen_cfi...  
1  https://www.allocine.fr/film/fichefilm_gen_cfi...  
2  https://www.allocine.fr/film/fichefilm_gen_cfi...  
3  https://www.allocine.fr/film/fichefilm_gen_cfi...  
4  https://www.allocine.fr/film/fichefilm_gen_cfi...  
5  https://www.allocine.fr/film/fichefilm_gen_cfi...  
6  https://www.allocine.fr/film/fichefilm_gen_cfi...  
7  https://www.allocine.fr/film/fichefilm_gen_cfi...  
8  https://www.allocine.fr/film/fichefilm_gen_cfi...  
9  https://www.allocine.fr/film

---
## Résumé Partie 1 : Code complet

In [10]:
# RÉSUMÉ PARTIE 1 : Code complet

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Configuration
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
tous_titres = []
tous_liens = []

# Boucle sur 3 pages
for page in range(1, 4):
    print(f'Page {page}...')
    
    url = f'https://www.allocine.fr/film/aucinema/?page={page}'
    response = requests.get(url, headers=headers)
    
    if not response.ok:
        continue
    
    soup = BeautifulSoup(response.text, 'lxml')
    section = soup.find('section', class_='section section-wrap gd-2-cols gd-gap-30')
    
    if not section:
        continue
    
    listes_ul = section.find_all('ul')
    if len(listes_ul) < 2:
        continue
    
    films_li = listes_ul[1].find_all('li', class_='mdl')
    
    for film in films_li:
        div_meta = film.find('div', class_='meta')
        if div_meta:
            lien_titre = div_meta.find('a', class_='meta-title-link')
            if lien_titre:
                tous_titres.append(lien_titre.get_text(strip=True))
                tous_liens.append('https://www.allocine.fr' + lien_titre['href'])
    
    time.sleep(1)

# Sauvegarder
df_films = pd.DataFrame({'titre': tous_titres, 'lien': tous_liens})
df_films.to_csv('films.csv', index=False)
print(f'Terminé ! {len(df_films)} films sauvegardés.')

Page 1...
Page 2...
Page 3...
Terminé ! 45 films sauvegardés.


---
# PARTIE 2 : Extraire les informations techniques des films

**Objectif** : Réimporter le CSV, choisir un film au hasard, et extraire ses informations techniques.

---
## Étape 2.1 : Importer le CSV et transformer en liste

**Ce qu'il faut faire :**
1. Lire le CSV avec `pd.read_csv('films.csv')`
2. Transformer la colonne `'lien'` en liste avec `.tolist()`
3. Afficher quelques liens pour vérifier

In [11]:
# Étape 2.1 : Importer CSV et créer la liste de liens

# Lire le CSV
df_import = pd.read_csv('films.csv')

print(f'Fichier importé : {len(df_import)} films')
print(df_import.head())

# Transformer la colonne 'lien' en liste
liste_liens = df_import['lien'].tolist()

print(f'\nType de liste_liens : {type(liste_liens)}')
print(f'Premiers liens :')
for lien in liste_liens[:3]:
    print(f'  - {lien}')

Fichier importé : 45 films
                titre                                               lien
0  L’Affaire Bojarski  https://www.allocine.fr/film/fichefilm_gen_cfi...
1  La Femme de ménage  https://www.allocine.fr/film/fichefilm_gen_cfi...
2  Le Mage du Kremlin  https://www.allocine.fr/film/fichefilm_gen_cfi...
3          Zootopie 2  https://www.allocine.fr/film/fichefilm_gen_cfi...
4              Gourou  https://www.allocine.fr/film/fichefilm_gen_cfi...

Type de liste_liens : <class 'list'>
Premiers liens :
  - https://www.allocine.fr/film/fichefilm_gen_cfilm=1000014092.html
  - https://www.allocine.fr/film/fichefilm_gen_cfilm=317403.html
  - https://www.allocine.fr/film/fichefilm_gen_cfilm=1000005026.html


---
## Étape 2.2 : Choisir un film au hasard et se connecter

**Ce qu'il faut faire :**
1. Choisir un lien au hasard
2. Se connecter à la page du film
3. Parser le HTML

In [23]:
# Étape 2.2 : Choisir un film et se connecter

# Choisir un lien au hasard
lien_film = liste_liens[1]
print(f'Film choisi : {lien_film}')

# Se connecter
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response_film = requests.get(lien_film, headers=headers)

if response_film.ok:
    print('Connexion réussie !')
    soup_film = BeautifulSoup(response_film.text, 'lxml')
else:
    print(f'Erreur : {response_film.status_code}')

Film choisi : https://www.allocine.fr/film/fichefilm_gen_cfilm=317403.html
Connexion réussie !


---
## Étape 2.3 : Trouver la section des informations techniques

**Ce qu'il faut faire :**
1. La section technique a la classe `"section ovw ovw-technical"`
2. Utiliser `soup.find('section', class_='section ovw ovw-technical')`
3. Afficher un extrait pour vérifier

In [13]:
# Étape 2.3 : Trouver la section technique

section_technique = soup_film.find('section', class_='section ovw ovw-technical')

if section_technique:
    print('Section technique trouvée !')
    print(section_technique.prettify()[:500])
else:
    print('Section technique non trouvée...')

Section technique trouvée !
<section class="section ovw ovw-technical">
 <div class="titlebar section-title">
  <h2 class="titlebar-title titlebar-title-md">
   Infos techniques
  </h2>
 </div>
 <div class="item">
  <span class="what light">
   Nationalité
  </span>
  <span class="that">
   <span class="ACrL2ZACrpbG1zL3BheXMtNTAwMi8= nationality">
    U.S.A.
   </span>
  </span>
 </div>
 <div class="item">
  <span class="what light">
   Distributeur
  </span>
  <span class="ACrL3NACrvY2lldGUvZmljaGVzb2NpZXRlLTEzMTIwLw== th


---
## Étape 2.4 : Extraire les éléments `<div class="item">`

**Ce qu'il faut faire :**
1. Chaque information technique est dans un `<div class="item">`
2. Utiliser `find_all('div', class_='item')` pour les récupérer tous
3. Afficher le premier pour voir la structure

In [14]:
# Étape 2.4 : Extraire les div item

items = section_technique.find_all('div', class_='item')
print(f'Nombre d\'items trouvés : {len(items)}')

# Afficher les 3 premiers
for i, item in enumerate(items[:3]):
    print(f'\n--- Item {i+1} ---')
    print(item.prettify())

Nombre d'items trouvés : 16

--- Item 1 ---
<div class="item">
 <span class="what light">
  Nationalité
 </span>
 <span class="that">
  <span class="ACrL2ZACrpbG1zL3BheXMtNTAwMi8= nationality">
   U.S.A.
  </span>
 </span>
</div>


--- Item 2 ---
<div class="item">
 <span class="what light">
  Distributeur
 </span>
 <span class="ACrL3NACrvY2lldGUvZmljaGVzb2NpZXRlLTEzMTIwLw== that blue-link">
  Paramount Pictures France
 </span>
</div>


--- Item 3 ---
<div class="item">
 <span class="what light">
  Année de production
 </span>
 <span class="that">
  2025
 </span>
</div>



---
## Étape 2.5 : Comprendre la structure variable des informations

**Point clé** : La structure HTML change selon que l'information contient un lien ou non !

**Sans lien :**
```html
<div class="item">
    <span class="what light">Année de production</span>
    <span class="that">2025</span>
</div>
```

**Avec lien :**
```html
<div class="item">
    <span class="what light">Distributeur</span>
    <a class="xXx that blue-link" href="...">Le Pacte</a>
</div>
```

**Solution** : Chercher d'abord `<span class="that">`, si absent chercher `<a>` avec `"that"` dans sa classe.

In [15]:
# Étape 2.5 : Afficher quelques items pour voir les différentes structures

for i, item in enumerate(items[:5]):
    nom = item.find('span', class_='what')
    span_that = item.find('span', class_='that')
    
    print(f'\nItem {i+1}:')
    print(f'  Nom: {nom.get_text(strip=True) if nom else "N/A"}')
    print(f'  A un <span class="that"> ? {span_that is not None}')
    
    if not span_that:
        # Chercher un <a> avec "that" dans les classes
        a_that = item.find('a', class_=lambda x: x and 'that' in x)
        print(f'  A un <a> avec "that" ? {a_that is not None}')


Item 1:
  Nom: Nationalité
  A un <span class="that"> ? True

Item 2:
  Nom: Distributeur
  A un <span class="that"> ? True

Item 3:
  Nom: Année de production
  A un <span class="that"> ? True

Item 4:
  Nom: Date de sortie DVD
  A un <span class="that"> ? True

Item 5:
  Nom: Date de sortie Blu-ray
  A un <span class="that"> ? True


---
## Étape 2.6 : Extraire le nom et la valeur d'un item

**Ce qu'il faut faire :**
1. Le nom est dans `<span class="what light">`
2. La valeur peut être dans :
   - `<span class="that">` (sans lien)
   - `<a class="... that ...">` (avec lien)
3. Tester d'abord `find('span', class_='that')`
4. Si None, chercher `find('a')` qui contient `'that'` dans ses classes

In [16]:
# Étape 2.6 : Extraire nom et valeur d'UN item (test)

# Prendre un item pour tester
item_test = items[0]

# Récupérer le nom
span_what = item_test.find('span', class_='what')
nom = span_what.get_text(strip=True) if span_what else None

# Récupérer la valeur - d'abord chercher span, sinon chercher a
valeur = None

# Méthode 1 : span class="that"
span_that = item_test.find('span', class_='that')
if span_that:
    valeur = span_that.get_text(strip=True)
else:
    # Méthode 2 : a avec "that" dans les classes
    a_that = item_test.find('a', class_=lambda x: x and 'that' in x)
    if a_that:
        valeur = a_that.get_text(strip=True)

print(f'Nom : {nom}')
print(f'Valeur : {valeur}')

Nom : Nationalité
Valeur : U.S.A.


---
## Étape 2.7 : Boucler sur tous les items et créer un dictionnaire

**Ce qu'il faut faire :**
1. Créer un dictionnaire vide `infos = {}`
2. Boucler sur tous les items
3. Pour chaque item :
   - Extraire le nom (span class="what")
   - Extraire la valeur (span ou a avec "that")
   - Ajouter au dictionnaire : `infos[nom] = valeur`

In [17]:
# Étape 2.7 : Boucler et créer le dictionnaire

infos = {}

for item in items:
    # Récupérer le nom
    span_what = item.find('span', class_='what')
    if not span_what:
        continue
    nom = span_what.get_text(strip=True)
    
    # Récupérer la valeur
    valeur = None
    
    # Essayer span class="that"
    span_that = item.find('span', class_='that')
    if span_that:
        valeur = span_that.get_text(strip=True)
    else:
        # Essayer a avec "that" dans les classes
        a_that = item.find('a', class_=lambda x: x and 'that' in x)
        if a_that:
            valeur = a_that.get_text(strip=True)
    
    # Ajouter au dictionnaire si on a nom et valeur
    if nom and valeur:
        infos[nom] = valeur

# Afficher le résultat
print('Informations techniques extraites :')
for cle, val in infos.items():
    print(f'  {cle} : {val}')

Informations techniques extraites :
  Nationalité : U.S.A.
  Distributeur : Paramount Pictures France
  Année de production : 2025
  Date de sortie DVD : -
  Date de sortie Blu-ray : -
  Date de sortie VOD : -
  Type de film : Long métrage
  Secrets de tournage : -
  Box Office France : 603 766 entrées
  Budget : -
  Langues : Anglais
  Format production : -
  Couleur : Couleur
  Format audio : -
  Format de projection : -
  N° de Visa : 165506


---
## Étape 2.8 : Convertir le dictionnaire en DataFrame

**Ce qu'il faut faire :**
1. Convertir en DataFrame avec `pd.DataFrame([infos])`
2. Afficher le résultat

In [18]:
# Étape 2.8 : Convertir en DataFrame

# Convertir en DataFrame (on met le dict dans une liste pour créer une ligne)
df_un_film = pd.DataFrame([infos])

print('DataFrame :')
print(df_un_film)

DataFrame :
  Nationalité               Distributeur Année de production  \
0      U.S.A.  Paramount Pictures France                2025   

  Date de sortie DVD Date de sortie Blu-ray Date de sortie VOD  Type de film  \
0                  -                      -                  -  Long métrage   

  Secrets de tournage Box Office France Budget  Langues Format production  \
0                   -   603 766 entrées      -  Anglais                 -   

   Couleur Format audio Format de projection N° de Visa  
0  Couleur            -                    -     165506  


---
## Étape 2.9 : Boucler sur plusieurs films

**Ce qu'il faut faire :**
1. Créer une liste vide `liste_infos = []`
2. Sélectionner quelques films (ex: les 5 premiers)
3. Boucler sur les liens :
   - Se connecter
   - Parser
   - Trouver la section technique
   - Extraire tous les items dans un dictionnaire
   - Ajouter le dictionnaire à la liste
4. **Important** : Ajouter `time.sleep(1)` entre chaque requête

In [19]:
# Étape 2.9 : Boucler sur plusieurs films

# Prendre les 5 premiers films pour tester
liens_a_scraper = liste_liens[:5]

# Liste pour stocker tous les dictionnaires
liste_infos = []

# Headers
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

for i, lien in enumerate(liens_a_scraper):
    print(f'Scraping film {i+1}/{len(liens_a_scraper)}...')
    
    # 1. Connexion
    response = requests.get(lien, headers=headers)
    
    if not response.ok:
        print(f'  → Erreur {response.status_code}')
        continue
    
    # 2. Parser
    soup = BeautifulSoup(response.text, 'lxml')
    
    # 3. Trouver la section technique
    section = soup.find('section', class_='section ovw ovw-technical')
    
    if not section:
        print(f'  → Section technique non trouvée')
        continue
    
    # 4. Extraire les items
    items = section.find_all('div', class_='item')
    
    # 5. Créer le dictionnaire pour ce film
    infos = {}
    infos['lien'] = lien  # Ajouter le lien pour référence
    
    for item in items:
        # Nom
        span_what = item.find('span', class_='what')
        if not span_what:
            continue
        nom = span_what.get_text(strip=True)
        
        # Valeur (span ou a)
        valeur = None
        span_that = item.find('span', class_='that')
        if span_that:
            valeur = span_that.get_text(strip=True)
        else:
            a_that = item.find('a', class_=lambda x: x and 'that' in x)
            if a_that:
                valeur = a_that.get_text(strip=True)
        
        if nom and valeur:
            infos[nom] = valeur
    
    # 6. Ajouter le dictionnaire à la liste
    liste_infos.append(infos)
    print(f'  → {len(infos)} informations récupérées')
    
    # 7. Pause entre les requêtes
    time.sleep(1)

print(f'\nTerminé ! {len(liste_infos)} films traités.')

Scraping film 1/5...
  → 17 informations récupérées
Scraping film 2/5...
  → 17 informations récupérées
Scraping film 3/5...
  → 18 informations récupérées
Scraping film 4/5...
  → 18 informations récupérées
Scraping film 5/5...
  → 16 informations récupérées

Terminé ! 5 films traités.


---
## Étape 2.10 : Créer le DataFrame final et exporter

**Ce qu'il faut faire :**
1. Convertir la liste de dictionnaires en DataFrame
2. Afficher le DataFrame
3. Exporter en CSV

In [20]:
# Étape 2.10 : Créer DataFrame et exporter

# Convertir la liste de dictionnaires en DataFrame
df_infos = pd.DataFrame(liste_infos)

print('DataFrame des informations techniques :')
print(df_infos)

# Afficher les colonnes disponibles
print(f'\nColonnes : {df_infos.columns.tolist()}')

# Exporter en CSV
df_infos.to_csv('films_infos_techniques.csv', index=False)
print('\nFichier "films_infos_techniques.csv" créé !')

DataFrame des informations techniques :
                                                lien     Nationalités  \
0  https://www.allocine.fr/film/fichefilm_gen_cfi...  France,Belgique   
1  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
2  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
3  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
4  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   

                     Distributeur Année de production Date de sortie DVD  \
0                        Le Pacte                2025                  -   
1         Metropolitan FilmExport                2025                  -   
2            Gaumont Distribution                2025                  -   
3  The Walt Disney Company France                2025                  -   
4                     StudioCanal                2025                  -   

  Date de sortie Blu-ray Date de sortie VOD  Type de film Secret

---
## Résumé Partie 2 : Code complet

In [21]:
# RÉSUMÉ PARTIE 2 : Code complet

import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# 1. Importer le CSV
df_import = pd.read_csv('films.csv')
liste_liens = df_import['lien'].tolist()

# 2. Configuration
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
liste_infos = []

# 3. Boucler sur les films (5 premiers pour l'exemple)
for lien in liste_liens[:5]:
    response = requests.get(lien, headers=headers)
    
    if not response.ok:
        continue
    
    soup = BeautifulSoup(response.text, 'lxml')
    section = soup.find('section', class_='section ovw ovw-technical')
    
    if not section:
        continue
    
    infos = {'lien': lien}
    
    for item in section.find_all('div', class_='item'):
        span_what = item.find('span', class_='what')
        if not span_what:
            continue
        nom = span_what.get_text(strip=True)
        
        span_that = item.find('span', class_='that')
        if span_that:
            valeur = span_that.get_text(strip=True)
        else:
            a_that = item.find('a', class_=lambda x: x and 'that' in x)
            valeur = a_that.get_text(strip=True) if a_that else None
        
        if nom and valeur:
            infos[nom] = valeur
    
    liste_infos.append(infos)
    time.sleep(1)

# 4. Créer et exporter le DataFrame
df_infos = pd.DataFrame(liste_infos)
df_infos.to_csv('films_infos_techniques.csv', index=False)
print(f'Terminé ! {len(df_infos)} films exportés.')

Terminé ! 5 films exportés.


---
# PARTIE 3 : Code final combiné

**Objectif** : Un script complet qui fait tout de A à Z.

In [22]:
# CODE FINAL COMBINÉ

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ============================================
# CONFIGURATION
# ============================================
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
NB_PAGES = 3
NB_FILMS_DETAILS = 5  # Nombre de films pour les détails

# ============================================
# PARTIE 1 : Récupérer la liste des films
# ============================================
print('=== PARTIE 1 : Récupération de la liste des films ===')

tous_titres = []
tous_liens = []

for page in range(1, NB_PAGES + 1):
    print(f'Page {page}/{NB_PAGES}...')
    
    url = f'https://www.allocine.fr/film/aucinema/?page={page}'
    response = requests.get(url, headers=headers)
    
    if not response.ok:
        continue
    
    soup = BeautifulSoup(response.text, 'lxml')
    section = soup.find('section', class_='section section-wrap gd-2-cols gd-gap-30')
    
    if not section:
        continue
    
    listes_ul = section.find_all('ul')
    if len(listes_ul) < 2:
        continue
    
    films_li = listes_ul[1].find_all('li', class_='mdl')
    
    for film in films_li:
        div_meta = film.find('div', class_='meta')
        if div_meta:
            lien_titre = div_meta.find('a', class_='meta-title-link')
            if lien_titre:
                tous_titres.append(lien_titre.get_text(strip=True))
                tous_liens.append('https://www.allocine.fr' + lien_titre['href'])
    
    time.sleep(1)

df_films = pd.DataFrame({'titre': tous_titres, 'lien': tous_liens})
df_films.to_csv('films.csv', index=False)
print(f'✓ {len(df_films)} films sauvegardés dans "films.csv"')

# ============================================
# PARTIE 2 : Récupérer les infos techniques
# ============================================
print('\n=== PARTIE 2 : Récupération des informations techniques ===')

df_import = pd.read_csv('films.csv')
liste_liens = df_import['lien'].tolist()[:NB_FILMS_DETAILS]

liste_infos = []

for i, lien in enumerate(liste_liens):
    print(f'Film {i+1}/{len(liste_liens)}...')
    
    response = requests.get(lien, headers=headers)
    
    if not response.ok:
        continue
    
    soup = BeautifulSoup(response.text, 'lxml')
    section = soup.find('section', class_='section ovw ovw-technical')
    
    if not section:
        continue
    
    infos = {'lien': lien}
    
    for item in section.find_all('div', class_='item'):
        span_what = item.find('span', class_='what')
        if not span_what:
            continue
        nom = span_what.get_text(strip=True)
        
        span_that = item.find('span', class_='that')
        if span_that:
            valeur = span_that.get_text(strip=True)
        else:
            a_that = item.find('a', class_=lambda x: x and 'that' in x)
            valeur = a_that.get_text(strip=True) if a_that else None
        
        if nom and valeur:
            infos[nom] = valeur
    
    liste_infos.append(infos)
    time.sleep(1)

df_infos = pd.DataFrame(liste_infos)
df_infos.to_csv('films_infos_techniques.csv', index=False)

# ============================================
# RÉSULTAT FINAL
# ============================================
print('\n' + '=' * 50)
print('RÉSULTAT FINAL')
print('=' * 50)
print(f'📋 Films listés : {len(df_films)}')
print(f'📊 Films avec détails : {len(df_infos)}')
print(f'📁 Fichiers créés : films.csv, films_infos_techniques.csv')
print('=' * 50)

# Aperçu
print('\nAperçu des informations techniques :')
print(df_infos.head())

=== PARTIE 1 : Récupération de la liste des films ===
Page 1/3...
Page 2/3...
Page 3/3...
✓ 45 films sauvegardés dans "films.csv"

=== PARTIE 2 : Récupération des informations techniques ===
Film 1/5...
Film 2/5...
Film 3/5...
Film 4/5...
Film 5/5...

RÉSULTAT FINAL
📋 Films listés : 45
📊 Films avec détails : 5
📁 Fichiers créés : films.csv, films_infos_techniques.csv

Aperçu des informations techniques :
                                                lien     Nationalités  \
0  https://www.allocine.fr/film/fichefilm_gen_cfi...  France,Belgique   
1  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
2  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
3  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   
4  https://www.allocine.fr/film/fichefilm_gen_cfi...              NaN   

                     Distributeur Année de production Date de sortie DVD  \
0                        Le Pacte                2025                  -   
1 

---
# Récapitulatif

| Concept | Description |
|---------|-------------|
| Pagination | Modifier `?page=N` dans l'URL |
| `time.sleep(1)` | Pause entre requêtes (politesse) |
| `df['col'].tolist()` | Convertir colonne en liste |
| Structure variable | Tester plusieurs sélecteurs (`span` puis `a`) |
| `class_=lambda x: x and 'that' in x` | Chercher une classe partielle |
| Liste de dicts → DataFrame | `pd.DataFrame(liste_de_dicts)` |

In [24]:
df_infos

,lien,Nationalités,Distributeur,Année de production,Date de sortie DVD,Date de sortie Blu-ray,Date de sortie VOD,Type de film,Secrets de tournage,Box Office France,Budget,Langues,Format production,Couleur,Format audio,Format de projection,N° de Visa,Nationalité,Récompense,Récompenses
0,https://www.allocine.fr/film/fichefilm_gen_cfi...,"France,Belgique",Le Pacte,2025,-,-,-,Long métrage,9 anecdotes,645 346 entrées,-,Français,-,Couleur,-,-,152664,NaN,NaN,NaN
1,https://www.allocine.fr/film/fichefilm_gen_cfi...,NaN,Metropolitan FilmExport,2025,-,-,-,Long métrage,10 anecdotes,3 823 351 entrées,-,Anglais,-,Couleur,-,-,165676,U.S.A.,NaN,NaN
2,https://www.allocine.fr/film/fichefilm_gen_cfi...,NaN,Gaumont Distribution,2025,-,-,-,Long métrage,9 anecdotes,330 014 entrées,-,"Anglais, Russe",-,Couleur,-,-,162964,France,1 nomination,NaN
3,https://www.allocine.fr/film/fichefilm_gen_cfi...,NaN,The Walt Disney Company France,2025,-,-,26/03/2026,Long métrage,14 anecdotes,8 164 429 entrées,-,Anglais,-,Couleur,-,-,166052,U.S.A.,NaN,4 nominations
4,https://www.allocine.fr/film/fichefilm_gen_cfi...,NaN,StudioCanal,2025,-,-,-,Long métrage,9 anecdotes,NaN,-,Français,-,Couleur,-,-,158213,France,NaN,NaN
